**Deep Learning, Visually** · Adapted from [Fundamentals of Deep Learning](https://github.com/NVDLI/fundamentals-of-deep-learning) by NVIDIA. Not affiliated with or endorsed by NVIDIA.

## License

This notebook is adapted from [Fundamentals of Deep Learning](https://github.com/NVDLI/fundamentals-of-deep-learning),
Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. NVIDIA's original material
keeps its licenses:

- Code cells: [Apache License 2.0](../../LICENSES/Apache-2.0.txt).
- Markdown and raw cells: [Creative Commons Attribution 4.0 International](../../LICENSES/CC-BY-4.0.txt).
- Third-party material keeps its original license, as listed in
  [`THIRD_PARTY_NOTICES.md`](../../THIRD_PARTY_NOTICES.md).

The changes made to NVIDIA's material are listed in [`CHANGES.md`](../../CHANGES.md).
NVIDIA names and logos are trademarks of NVIDIA and are not licensed. This course
is not affiliated with, sponsored or endorsed by NVIDIA.

**Additions.** The cells marked **Interactive**, the *Check Your Understanding*
and *Practice Exercises* sections, and the files in [`interactive/`](interactive/)
are Copyright (c) 2026 Pedro Gomes, all rights reserved. You may use them only
for your own personal, non-commercial learning, as described in
[`LICENSES/LicenseRef-DLV-Personal-Use.txt`](../../LICENSES/LicenseRef-DLV-Personal-Use.txt).
You may not share, publish or sell them.

# 1. Image Classification with the MNIST Dataset

In this section, we will do the "Hello World" of deep learning: training a deep learning model to correctly classify hand-written digits.

## 1.1 Setup for Google Colab

**Only run this cell if you are using Google Colab.** In other environments, skip it and continue with the lesson.

In [ ]:
%%bash
# @title Setup for Google Colab { display-mode: "form" }
# @markdown **Only run this cell if you are using Google Colab.** Otherwise, skip it. Select a GPU under **Runtime → Change runtime type** before running it.
set -e

# Skip setup outside Google Colab.
if [ -z "${COLAB_RELEASE_TAG:-}${COLAB_BACKEND_VERSION:-}" ]; then
    exit 0
fi

# Download only the support files used by this lab.
course_url=https://raw.githubusercontent.com/OblackatO/deep-learning-visually/main/course_content

# Download only this lab's support files; preserve existing files and student work.
for course_file in interactive/mnist.py interactive/quiz.py; do
    if [ ! -e "$course_file" ]; then
        mkdir -p "$(dirname "$course_file")"
        curl -fsSL "$course_url/tutorials/$course_file" -o "$course_file.download"
        mv "$course_file.download" "$course_file"
    fi
done

# Install only this lab's packages that are missing from the current runtime.
missing_packages=()
for course_package in torch torchvision matplotlib ipywidgets; do
    if ! pip show "$course_package" >/dev/null 2>&1; then
        missing_packages+=("$course_package")
    fi
done
if [ "${#missing_packages[@]}" -gt 0 ]; then
    pip install --quiet "${missing_packages[@]}"
fi

echo "Course ready. Continue with the next cell."


## 1.2 Objectives

* Understand how deep learning can solve problems traditional programming methods cannot
* Learn about the [MNIST handwritten digits dataset](http://yann.lecun.com/exdb/mnist/)
* Use the [torchvision](https://pytorch.org/vision/stable/index.html) to load the MNIST dataset and prepare it for training
* Create a simple neural network to perform image classification
* Train the neural network using the prepped MNIST dataset
* Observe the performance of the trained neural network

Let's begin by loading the libraries used in this notebook:

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

# Visualization tools
import torchvision
import torchvision.transforms.v2 as transforms
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt

This notebook also has **interactive diagrams**: sliders and buttons that update a picture as you move them. They are defined in [`interactive/mnist.py`](interactive/mnist.py) and never change your data or model. Run the cell below to load them.

In [ ]:
from interactive import mnist as viz

In PyTorch, we can use our GPU in our operations by setting the [device](https://pytorch.org/docs/stable/tensor_attributes.html#torch.device) to `cuda`. The function `torch.cuda.is_available()` will confirm PyTorch can recognize the GPU.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

### 1.2.1 The Problem: Image Classification

In traditional programming, the programmer is able to articulate rules and conditions in their code that their program can then use to act in the correct way. This approach continues to work exceptionally well for a huge variety of problems.

Image classification, which asks a program to correctly classify an image it has never seen before into its correct class, is near impossible to solve with traditional programming techniques. How could a programmer possibly define the rules and conditions to correctly classify a huge variety of images, especially taking into account images that they have never seen?

### 1.2.2 The Solution: Deep Learning

Deep learning excels at pattern recognition by trial and error. By training a deep neural network with sufficient data, and providing the network with feedback on its performance via training, the network can identify, though a huge amount of iteration, its own set of conditions by which it can act in the correct way.

**Interactive: the difference in one picture.** In traditional programming, a person writes the rules. In deep learning, the model learns the rules from examples and their correct answers.

In [ ]:
viz.show_rules_vs_learning()

## 1.3 The MNIST Dataset

In the history of deep learning, the accurate image classification of the [MNIST dataset](http://yann.lecun.com/exdb/mnist/), a collection of 70,000 grayscale images of handwritten digits from 0 to 9, was a major development. While today the problem is considered trivial, doing image classification with MNIST has become a kind of "Hello World" for deep learning.

Here are 40 of the images included in the MNIST dataset:

<img src="https://raw.githubusercontent.com/OblackatO/deep-learning-visually/main/course_content/tutorials/images/mnist1.png" style="width: 600px;">

### 1.3.1 Training and Validation Data and Labels

When working with images for deep learning, we need both the images themselves, usually denoted as `X`, and also, correct [labels](https://developers.google.com/machine-learning/glossary#label) for these images, usually denoted as `Y`. Furthermore, we need `X` and `Y` values both for *training* the model, and then, a separate set of `X` and `Y` values for *validating* the performance of the model after it has been trained.

We can imagine these `X` and `Y` pairs as a set of flash cards. A student can train with one set of flashcards, and to validate the student learned the correct concepts, a teacher might quiz the student with a different set of flash cards. 

Therefore, we need 4 segments of data for the MNIST dataset:

1. `x_train`: Images used for training the neural network
2. `y_train`: Correct labels for the `x_train` images, used to evaluate the model's predictions during training
3. `x_valid`: Images set aside for validating the performance of the model after it has been trained
4. `y_valid`: Correct labels for the `x_valid` images, used to evaluate the model's predictions after it has been trained

The process of preparing data for analysis is called [Data Engineering](https://medium.com/@rchang/a-beginners-guide-to-data-engineering-part-i-4227c5c457d7). To learn more about the differences between training data and validation data (as well as test data), check out [this article](https://machinelearningmastery.com/difference-test-validation-datasets/) by Jason Brownlee.

### 1.3.2 Loading the Data Into Memory (with TorchVision)

There are many [deep learning frameworks](https://developer.nvidia.com/deep-learning-frameworks), each with their own merits. In this workshop we will be working with [PyTorch 2](https://pytorch.org/get-started/pytorch-2.0/), and specifically with the [Sequential API](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html). The Sequential API has many useful built in functions designed for constructing neural networks. It is also a legitimate choice for deep learning in a professional setting due to its [readability](https://blog.pragmaticengineer.com/readable-code/) and efficiency, though it is not alone in this regard, and it is worth investigating a variety of frameworks when beginning a deep learning project.

We will also use the [TorchVision](https://pytorch.org/vision/stable/index.html) library. One of the many helpful features that it provides are modules containing helper methods for [many common datasets](https://pytorch.org/vision/main/datasets.html), including MNIST.

We will begin by loading both the `train` and `valid` datasets for [MNIST](https://pytorch.org/vision/main/generated/torchvision.datasets.MNIST.html#torchvision.datasets.MNIST).

In [ ]:
train_set = torchvision.datasets.MNIST("./data/", train=True, download=True)
valid_set = torchvision.datasets.MNIST("./data/", train=False, download=True)

We stated above that the MNIST dataset contained 70,000 grayscale images of handwritten digits. By executing the following cells, we can see that TorchVision has partitioned 60,000 of these [PIL Images](https://pillow.readthedocs.io/en/stable/reference/Image.html) for training, and 10,000 for validation (after training).

In [ ]:
train_set

In [ ]:
valid_set

*Note*: The `Split` for `valid_set` is stated as `Test`, but we will be using the data for validation in our hands-on exercises. To learn more about the difference between `Train`, `Valid`, and `Test` datasets, please view [this article](https://kili-technology.com/training-data/training-validation-and-test-sets-how-to-split-machine-learning-data) by Kili.

### 1.3.3 Exploring the MNIST Data

Let's take the first x, y pair from `train_set` and review the data structures:

In [ ]:
x_0, y_0 = train_set[0]

In [ ]:
x_0

In [ ]:
type(x_0)

Is this a 5 or a poorly written 3? We can view the corresponding label to be sure.

In [ ]:
y_0

In [ ]:
type(y_0)

**Interactive: browse the dataset.** Switch between the training and validation sets and move the slider to look at any image `x` and its label `y`. The bar chart shows how many images of each digit there are. The digit of the current image is highlighted.

In [ ]:
viz.explore_dataset(train_set, valid_set)

## 1.4 Tensors

If a vector is a 1-dimensional array, and a matrix is a 2-dimensional array, a tensor is an n-dimensional array representing any number of dimensions. Most modern neural network frameworks are powerful tensor processing tools.

One example of a 3-dimensional tensor could be pixels on a computer screen. The different dimensions would be width, height, and color channel. Video games use matrix mathematics to calculate pixel values in a similar way to how neural networks calculate tensors. This is why GPUs are effective tensor processesing machines.

Let's convert our images into tensors so we can later process them with a neural network. TorchVision has a useful function to convert [PIL Images](https://pillow.readthedocs.io/en/stable/reference/Image.html) into tensors with the [ToImage](https://pytorch.org/vision/main/generated/torchvision.transforms.v2.ToImage.html) class, followed by the [ToDtype](https://pytorch.org/vision/main/generated/torchvision.transforms.v2.ToDtype.html) class to convert the pixel values to floats:

In [ ]:
trans = transforms.Compose([transforms.ToImage(), transforms.ToDtype(torch.float32, scale=True)])
x_0_tensor = trans(x_0)

[PyTorch tensors](https://pytorch.org/docs/stable/tensors.html#torch.Tensor) have a number of useful properies and methods. We can verify the data type:

In [ ]:
x_0_tensor.dtype

We can verify the minimum and maximum values. PIL Images have a potential integer range of [0, 255], but [ToDtype](https://pytorch.org/vision/main/generated/torchvision.transforms.v2.ToDtype.html) with `scale=True` converts it to a float range of [0.0, 1.0].

In [ ]:
x_0_tensor.min()

In [ ]:
x_0_tensor.max()

We can also view the size of each dimension. PyTorch uses a `C x H x W` convention, which means the first dimension is color channel, the second is height, and the third is width.

Since these images are black and white, there is only 1 color channel. The images are square being 28 pixels tall and wide:

In [ ]:
x_0_tensor.size()

We can also look at the values directly:

In [ ]:
x_0_tensor

**Interactive: look inside the tensor.** Each of these 784 numbers is the brightness of one pixel: 0.0 is black and 1.0 is white. Move the sliders to pick a pixel (red square). The right panel zooms in on the 7 × 7 pixels around it and prints their values.

In [ ]:
viz.explore_pixels(x_0_tensor)

By default, a tensor is processed with a [CPU](https://www.arm.com/glossary/cpu).

In [ ]:
x_0_tensor.device

To move it to a GPU, we can use the `.cuda` method.

In [ ]:
x_0_gpu = x_0_tensor.cuda()
x_0_gpu.device

The `.cuda` method will fail if a GPU is not recognized by PyTorch. In order to make our code flexible, we can send our tensor `to` the `device` we identified at the start of this notebook. This way, our code will run much faster if a GPU is available, but the code will not break if there is no available GPU.

In [ ]:
x_0_tensor.to(device).device

Sometimes, it can be hard to interpret so many numbers. Thankfully, TorchVision can convert `C x H x W` tensors back into a PIL image with the [to_pil_image](https://pytorch.org/vision/main/generated/torchvision.transforms.functional.to_pil_image.html) function.

In [ ]:
image = F.to_pil_image(x_0_tensor)
plt.imshow(image, cmap='gray')

## 1.5 Preparing the Data for Training

Earlier, we created a `trans` variable to convert an image to a tensor. [Transforms](https://pytorch.org/vision/stable/transforms.html) are a group of torchvision functions that can be used to transform a dataset.

### 1.5.1 Transforms

The [Compose](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.Compose.html#torchvision.transforms.v2.Compose) fuction combines a list of transforms. We will learn more about transforms in a later notebook, but have copied the `trans` definition below as an introduction.

In [ ]:
trans = transforms.Compose([transforms.ToImage(), transforms.ToDtype(torch.float32, scale=True)])

Before, we only applied `trans` to one value. There are multiple ways we can apply our list of transforms to a dataset. One such way is to set it to a dataset's `transform` variable. 

In [ ]:
train_set.transform = trans
valid_set.transform = trans

### 1.5.2 DataLoaders

If our dataset is a deck of flash cards, a [DataLoader](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html#preparing-your-data-for-training-with-dataloaders) defines how we pull cards from the deck to train an AI model. We could show our models the entire dataset at once. Not only does this take a lot of computational resources, but [research shows](https://arxiv.org/pdf/1804.07612) using a smaller batch of data is more efficient for model training.

For example, if our `batch_size` is 32, we will train our model by shuffling the deck and drawing 32 cards. We do not need to shuffle for validation as the model is not learning, but we will still use a `batch_size` to prevent memory errors.

The batch size is something the model developer decides, and the best value will depend on the problem being solved. Research shows 32 or 64 is sufficient for many machine learning problems and is the default in some machine learning frameworks, so we will use 32 here.

In [ ]:
batch_size = 32

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)

**Interactive: shuffled vs not shuffled.** The button draws the first batch from each `DataLoader`. Click it a few times: the training batch changes every time because `shuffle=True`, while the validation batch stays the same. The number in the corner of each image is its label.

In [ ]:
viz.explore_batches(train_loader, valid_loader)

## 1.6 Creating the Model

It's time to build the model! Neural networks are composed of layers where each layer performs a mathematical operation on the data it receives before passing it to the next layer. To start, we will create a "Hello World" level model made from 4 components:

1. A [Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html) used to convert n-dimensional data into a vector. 
2. An input layer, the first layer of neurons
3. A hidden layer, another layor of neurons "hidden" between the input and output
4. An output layer, the last set of neurons which returns the final prediction from the model

More information about these layers is available in [this blog post](https://medium.com/@sarita_68521/basic-understanding-of-neural-network-structure-eecc8f149a23) by Sarita.

Let's create a `layers` variable to hold our list of layers.

**Interactive: the path of one image through the model.** Click a step to see what happens there and how many numbers come out. We will build these layers one by one in the next cells.

In [ ]:
viz.show_pipeline(x_0_tensor)

In [ ]:
layers = []
layers

### 1.6.1 Flattening the Image

When we looked at the shape of our data above, we saw the images had 3 dimensions: `C x H x W`. To flatten an image means to combine all of these images into 1 dimension. Let's say we have a tensor like the one below. Try running the code cell to see what it looks like before and after being flattened.

In [ ]:
test_matrix = torch.tensor(
    [[1, 2, 3],
     [4, 5, 6],
     [7, 8, 9]]
)
test_matrix

In [ ]:
nn.Flatten()(test_matrix)

Nothing happened? That's because neural networks expect to recieve a batch of data. Currently, the Flatten layer sees three vectors as opposed to one 2d matrix. To fix this, we can "batch" our data by adding an extra pair of brackets. Since `test_matrix` is now a tensor, we can do that with the shorthand below. `None` adds a new dimension where `:` selects all the data in a tensor.

In [ ]:
batch_test_matrix = test_matrix[None, :]
batch_test_matrix

In [ ]:
nn.Flatten()(batch_test_matrix)

Order matters! This is what happens when we do it the other way:

In [ ]:
nn.Flatten()(test_matrix[:, None])

**Interactive: flattening a real image.** `nn.Flatten()` lays the 28 rows of the image end to end. Move the slider to pick a row and see where its 28 values end up in the list of 784 numbers.

In [ ]:
viz.explore_flatten(x_0_tensor)

Now that we've gotten the hang of the `Flatten` layer, let's add it to our list of `layers`.

In [ ]:
layers = [
    nn.Flatten()
]
layers

### 1.6.2 The Input Layer

Our first layer of neurons connects our flattened image to the rest of our model. To do that, we will use a [Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) layer. This layer will be *densely connected*, meaning that each neuron in it, and its weights, will affect every neuron in the next layer.

In order to create these weights, Pytorch needs to know the size of our inputs and how many neurons we want to create. 
Since we've flattened our images, the size of our inputs is the number of channels, number of pixels vertically, and number of pixels horizontally multiplied together.

In [ ]:
input_size = 1 * 28 * 28

Choosing the correct number of neurons is what puts the "science" in "data science" as it is a matter of capturing the statistical complexity of the dataset. For now, we will use `512` neurons. Try playing around with this value later to see how it affects training and to start developing a sense for what this number means.

We will learn more about activation functions later, but for now, we will use the [relu](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html) activation function, which in short, will help our network to learn how to make more sophisticated guesses about data than if it were required to make guesses based on some strictly linear function.

In [ ]:
layers = [
    nn.Flatten(),
    nn.Linear(input_size, 512),  # Input
    nn.ReLU(),  # Activation for input
]
layers

**Interactive: what one neuron computes.** Each of the 512 neurons in a `Linear` layer multiplies every input by a weight, adds the results, and adds a bias. ReLU then replaces a negative result with 0. This neuron has only 3 inputs instead of 784, so you can see every number. Try making `z` negative.

In [ ]:
viz.explore_neuron()

### 1.6.3 The Hidden Layer

Now we will add an additional densely connected linear layer. We will cover why adding another set of neurons can help improve learning in the next lesson. Just like how the input layer needed to know the shape of the data that was being passed to it, a hidden layer's [nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) needs to know the shape of the data being passed to it. Each neuron in the previous layer will compute one number, so the number of inputs into the hidden layer is the same as the number of neurons in the previous later.

In [ ]:
layers = [
    nn.Flatten(),
    nn.Linear(input_size, 512),  # Input
    nn.ReLU(),  # Activation for input
    nn.Linear(512, 512),  # Hidden
    nn.ReLU()  # Activation for hidden
]
layers

### 1.6.4 The Output Layer

Finally, we will add an output layer. In this case, since the network is to make a guess about a single image belonging to 1 of 10 possible categories, there will be 10 outputs. Each output is assigned a neuron. The larger the value of the output neuron compared to the other neurons, the more the model predicts the input image belongs to the output neuron's assigned class.

We will not assign the `relu` function to the output layer. Instead, we will apply a `loss function` covered in the next section.

In [ ]:
n_classes = 10

layers = [
    nn.Flatten(),
    nn.Linear(input_size, 512),  # Input
    nn.ReLU(),  # Activation for input
    nn.Linear(512, 512),  # Hidden
    nn.ReLU(),  # Activation for hidden
    nn.Linear(512, n_classes)  # Output
]
layers

**Interactive: the whole network.** Every weight and bias in this diagram is a number the model learns during training. Move the sliders to see how the number of neurons and layers changes the size of the model.

In [ ]:
viz.show_network(input_size, n_classes)

### 1.6.5 Compiling the Model

A [Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) model expects a sequence of arguments, not a list, so we can use the [* operator](https://docs.python.org/3/reference/expressions.html#expression-lists) to unpack our list of layers into a sequence. We can print the model to verify these layers loaded correctly.

In [ ]:
model = nn.Sequential(*layers)
model

Much like tensors, when the model is first initialized, it will be processed on a CPU. To have it process with a GPU, we can use `to(device)`.

In [ ]:
model.to(device)

To check which device a model is on, we can check which device the model parameters are on. Check out this [stack overflow](https://stackoverflow.com/questions/58926054/how-to-get-the-device-type-of-a-pytorch-module-conveniently) post for more information.

In [ ]:
next(model.parameters()).device

[PyTorch 2.0](https://pytorch.org/get-started/pytorch-2.0/) introduced the ability to compile a model for faster performance. Learn more about it [here](https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html).

In [ ]:
model = torch.compile(model)

## 1.7 Training the Model

Now that we have prepared training and validation data, and a model, it's time to train our model with our training data, and verify it with its validation data.

"Training a model with data" is often also called "fitting a model to data." Put another way, it highlights that the shape of the model changes over time to more accurately understand the data that it is being given.

### 1.7.1 Loss and Optimization

Just like how teachers grade students, we need to provide the model a function in which to grade its answers. This is called a `loss function`. We will use a type of loss function called [CrossEntropy](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) which is designed to grade if a model predicted the correct category from a group of categories.

In [ ]:
loss_function = nn.CrossEntropyLoss()

**Interactive: how the loss grades an answer.** The model outputs 10 scores. Cross entropy turns them into probabilities (softmax) and takes −ln of the probability given to the correct digit. Drag the scores: the loss is low when the model is confident and right, and high when it is confident and wrong.

In [ ]:
viz.explore_loss()

Next, we select an `optimizer` for our model. If the `loss_function` provides a grade, the optimizer tells the model how to learn from this grade to do better next time.

In [ ]:
optimizer = Adam(model.parameters())

**Interactive: how the optimizer learns.** The optimizer moves each weight a small step in the direction that lowers the loss. Here is the idea with a single weight. Try a learning rate above 1, then above 2. `Adam` adjusts the step size for each weight automatically, but the principle is the same for all 669,706 weights and biases of our model.

In [ ]:
viz.explore_gradient_descent()

### 1.7.2 Calculating Accuracy

While the results of the loss function are effective in helping our model learn, the values can be difficult to interpret for humans. This is why data scientists often include other metrics like accuracy.

In order to accurately calculate accuracy, we should compare the number of correct classifications compared to the total number of predictions made. Since we're showing data to the model in batches, our accuracy can be calculated along with these batches.

First, the total number of predictions is the same size as our dataset. Let's assign the size of our datasets to `N` where `n` is synonymous with the `batch size`. 

In [ ]:
train_N = len(train_loader.dataset)
valid_N = len(valid_loader.dataset)

Next, we'll make a function to calculate the accuracy for each batch. The result is a fraction of the total accuracy, so we can add the accuracy of each batch together to get the total.

In [ ]:
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N

### 1.7.3 The Train Function

Here is where everything comes together. Below is the function we've defined to train our model based on the training data. We will walk through each line of code in more detail later, but take a moment to review how it is structured. Can you recognize the variables we created earlier?

In [ ]:
def train():
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        output = model(x)
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

**Interactive: one training step at a time.** Press play, or move the slider, to step through what `train()` does with every batch. The matching lines of code are highlighted.

In [ ]:
viz.show_training_loop(len(train_loader))

### 1.7.4 The Validate Function

Similarly, this is the code for validating the model with data it did not train on. Can you spot some differences with the `train` function?

In [ ]:
def validate():
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            output = model(x)

            loss += loss_function(output, y).item()
            accuracy += get_batch_accuracy(output, y, valid_N)
    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

### 1.7.5 The Training Loop

To see how the model is progressing, we will alternated between training and validation. Just like how it might take a student a few times going through their deck of flash cards to learn all the concepts, the model will go through the training data multiple times to get a better and better understanding.

An `epoch` is one complete pass through the entire dataset. Let's train and validate the model for 5 `epochs` to see how it learns.

In [ ]:
epochs = 5

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train()
    validate()

We're already close to 100%! Let's see if it's true by testing it on our original sample. We can use our model like a function:

In [ ]:
prediction = model(x_0_tensor.to(device))
prediction

There should be ten numbers, each corresponding to a different output neuron. Thanks to how the data is structured, the index of each number matches the corresponding handwritten number. The 0th index is a prediction for a handwritten 0, the 1st index is a prediction for a handwritten 1, and so on.

We can use the `argmax` function to find the index of the highest value.

In [ ]:
prediction.argmax(dim=1, keepdim=True)

Did it get it right?

In [ ]:
y_0

**Interactive: check the trained model.** Browse the model's predictions on the 10,000 validation images. Tick *Only show mistakes* to see which digits it gets wrong. Some of them are hard to read even for people.

In [ ]:
viz.explore_predictions(model, valid_set, device)

## 1.8 Summary

The model did quite well! The accuracy quickly reached close to 100%, as did the validation accuracy. We now have a model that can be used to accurately detect and classify hand-written images.

The next step would be to use this model to classify new not-yet-seen handwritten images. This is called [inference](https://blogs.nvidia.com/blog/2016/08/22/difference-deep-learning-training-inference-ai/). We'll explore the process of inference in a later exercise.

It's worth taking a moment to appreciate what we've done here. Historically, the expert systems that were built to do this kind of task were extremely complicated, and people spent their careers building them (check out the references on the [official MNIST page](http://yann.lecun.com/exdb/mnist/) and the years milestones were reached).

MNIST is not only useful for its historical influence on Computer Vision, but it's also a great [benchmark](http://www.cs.toronto.edu/~serailhydra/publications/tbd-iiswc18.pdf) and debugging tool. Having trouble getting a fancy new machine learning architecture working? Check it against MNIST. If it can't learn on this dataset, chances are it won't learn on more complicated images and datasets.

## 1.9 Check Your Understanding

These questions are in the style of professional certification exams: multiple choice, one correct answer, often with a short scenario. Pick an answer and click **Check** to see the explanation. Your score is shown at the top.

In [ ]:
viz.quiz()

## 1.10 Practice Exercises

Time to write some code yourself. Each exercise has a cell to complete (replace the `None` values and `TODO` comments), a cell that checks your answer, and a hidden solution. Try it on your own before you open the solution.

### Exercise 1: Count the parameters

Earlier we worked out that our model has 669,706 weights and biases. Now let PyTorch count them. Every tensor in `model.parameters()` has a [numel()](https://pytorch.org/docs/stable/generated/torch.numel.html) method that returns how many values it holds. Store the total in `n_params`.

In [ ]:
n_params = None  # TODO: the total number of values in all of model.parameters()
n_params

In [ ]:
viz.check_parameter_count(n_params, model)

<details>
<summary><b>Show solution</b></summary>

```python
n_params = sum(p.numel() for p in model.parameters())
n_params
```

</details>

### Exercise 2: Predict a whole batch

Use the model to predict the digit of every image in the first validation batch.

1. Run the model on `x`. Remember to move `x` to `device` first.
2. `preds`: the predicted digit of each image. Use `argmax`, like in section 1.7.5.
3. `batch_acc`: the fraction of images whose prediction matches the label in `y`, as a number between 0 and 1.

In [ ]:
x, y = next(iter(valid_loader))

preds = None      # TODO: the predicted digit for each image in the batch
batch_acc = None  # TODO: the fraction of images in the batch that were predicted correctly

In [ ]:
viz.check_batch_predictions(preds, batch_acc, model, valid_loader)

<details>
<summary><b>Show solution</b></summary>

```python
x, y = next(iter(valid_loader))
output = model(x.to(device))
preds = output.argmax(dim=1).cpu()
batch_acc = (preds == y).float().mean().item()
```

</details>

### Exercise 3: A smaller model and your own training function

**Part A.** Build `small_model`, a model with a single layer of 128 neurons: `Flatten` → `Linear` (784 → 128) → `ReLU` → `Linear` (128 → 10).

In [ ]:
small_model = None  # TODO: nn.Sequential(...)

In [ ]:
viz.check_small_model(small_model)

**Part B.** Write `train_one_epoch`, a reusable version of the `train` function from section 1.7.3. It receives a model, a `DataLoader`, an optimizer and a loss function, trains for one epoch, and returns the **average loss per batch**.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_function):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # TODO: 1. predict  2. clear the gradients  3. compute the loss
        #       4. backward pass  5. update the weights  6. add the loss to total_loss
        ...
    return None  # TODO: the average loss per batch

In [ ]:
viz.check_train_one_epoch(train_one_epoch, train_set, device)

<details>
<summary><b>Show solution</b></summary>

```python
small_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)


def train_one_epoch(model, loader, optimizer, loss_function):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        output = model(x)
        optimizer.zero_grad()
        loss = loss_function(output, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)
```

</details>

**Part C.** Train `small_model` for 2 epochs with your function, then compare its validation accuracy with the original model. It has 85% fewer parameters. Is it much worse?

In [ ]:
assert small_model is not None, "Complete Part A first."
small_model = small_model.to(device)
small_optimizer = Adam(small_model.parameters())

for epoch in range(2):
    average_loss = train_one_epoch(small_model, train_loader, small_optimizer, loss_function)
    print(f"Epoch {epoch}: average loss {average_loss:.4f}")

print(f"Original model: {viz.accuracy(model, valid_loader):.2%} validation accuracy")
print(f"Small model:    {viz.accuracy(small_model, valid_loader):.2%} validation accuracy")

### Exercise 4 (challenge): Which digit is the hardest?

Compute the validation accuracy of the original `model` for each digit separately. For each digit `d`, that is the number of validation images labelled `d` that the model predicts as `d`, divided by the number of validation images labelled `d`. Store the 10 values in the list `per_digit_acc`.

Then scroll back to the prediction explorer in section 1.7.5, tick *Only show mistakes*, and look for examples of the hardest digit. Can you see why the model struggles with them?

In [ ]:
correct = [0] * 10
total = [0] * 10

model.eval()
with torch.no_grad():
    for x, y in valid_loader:
        preds = model(x.to(device)).argmax(dim=1).cpu()
        # TODO: for every digit d, add to total[d] and correct[d]

per_digit_acc = None  # TODO: a list of 10 fractions, one per digit
per_digit_acc

In [ ]:
viz.check_per_digit_accuracy(per_digit_acc, model, valid_loader)

<details>
<summary><b>Show solution</b></summary>

```python
correct = [0] * 10
total = [0] * 10

model.eval()
with torch.no_grad():
    for x, y in valid_loader:
        preds = model(x.to(device)).argmax(dim=1).cpu()
        for d in range(10):
            mask = y == d
            total[d] += mask.sum().item()
            correct[d] += (preds[mask] == d).sum().item()

per_digit_acc = [c / t for c, t in zip(correct, total)]
per_digit_acc
```

</details>

### 1.7.1 Clear the Memory

Before moving on, please execute the following cell to clear up the GPU memory. This is required to move on to the next notebook.

In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

### 1.7.2 Next

In this section you learned how to build and train a simple neural network for image classification. In the next section, you will be asked to build your own neural network and perform data preparation to solve a different image classification problem.